In [23]:
import pandas as pd
import os
import numpy as np
from pandas.plotting import scatter_matrix
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline, FeatureUnion

from sklearn import set_config
set_config(transform_output = "default")

%matplotlib inline
import matplotlib.pyplot as plt
def load_data():
    csv_path = os.path.join("datasets", "student_exam_performance.csv")
    return pd.read_csv(csv_path) 
stud_ex_perf = load_data()

stud_ex_perf["performance_grade"] = stud_ex_perf["performance_grade"].replace(stud_ex_perf["performance_grade"].unique(), [5.0, 4.0, 3.0, 2.0, 2.0])
stud_ex_perf = stud_ex_perf[~stud_ex_perf["gender"].isin(["Other"])]
stud_ex_perf = stud_ex_perf[~stud_ex_perf["school_type"].isin(["Charter"])]
stud_ex_perf = stud_ex_perf.reset_index(drop=True)  

split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_index, test_index in split.split(stud_ex_perf, stud_ex_perf[["gender"]]):
    strat_train_set = stud_ex_perf.loc[train_index]
    strat_test_set = stud_ex_perf.loc[test_index] 

def data_split_at_models(data_train, data_test, columns_to_remove):
    for i in columns_to_remove:
        data_train = data_train.drop(i, axis=1)
        data_test = data_test.drop(i, axis=1)
    return data_train, data_test

train_exam_score, test_exam_score = data_split_at_models(strat_train_set, strat_test_set, ["performance_grade", "pass_status", "performance_level"])
train_pef_grade, test_perf_grade = data_split_at_models(strat_train_set, strat_test_set, ["exam_score", "pass_status", "performance_level"])
train_pass_status, test_pass_status = data_split_at_models(strat_train_set, strat_test_set, ["performance_grade", "exam_score", "performance_level"])
train_perf_level, test_perf_level = data_split_at_models(strat_train_set, strat_test_set, ["performance_grade", "pass_status", "exam_score"])

corr_matrix = train_exam_score.corr(numeric_only=True, method="pearson") #!spearman helps to avoid emissions!
corr_matrix["exam_score"].sort_values(ascending=False)

X_exam_score = train_exam_score.drop("exam_score", axis=1)
y_exam_score = train_exam_score["exam_score"]

num_attr = list(X_exam_score.select_dtypes(include='number'))
cat_attr = list(X_exam_score.select_dtypes(exclude='number'))

class DataFrameSelector(BaseEstimator, TransformerMixin):
    def __init__(self, attribute_names) :
        self.attribute_names = attribute_names
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        return X[self.attribute_names].values 

num_pipeline = Pipeline([
    ('selector', DataFrameSelector(num_attr)),
    ('imputer', SimpleImputer(strategy="median")),
    ('std_scaler', StandardScaler()),
])

cat_pipeline = Pipeline([
    ('selector', DataFrameSelector(cat_attr)),
    ('cat_encoder', OneHotEncoder()),
])

full_pipeline = FeatureUnion(transformer_list=[
    ("num_pipeline", num_pipeline),
    ("cat_pipeline", cat_pipeline),
])

X_exam_score_prepared = full_pipeline.fit_transform(X_exam_score)

print(X_exam_score_prepared)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 2822560 stored elements and shape (70564, 70646)>
  Coords	Values
  (0, 0)	-1.0010667144203496
  (0, 1)	0.10148773443468213
  (0, 2)	-0.04850292651266028
  (0, 3)	0.013991061925725098
  (0, 4)	-1.6063437805247722
  (0, 5)	-0.38193001971851054
  (0, 6)	-0.24093122056010186
  (0, 7)	-0.6546492528797243
  (0, 8)	0.11360196551575605
  (0, 9)	-0.6049488115071248
  (0, 10)	-0.5278262248412238
  (0, 11)	0.8687690196372682
  (0, 12)	-0.7984875663469704
  (0, 13)	0.5756692830709793
  (0, 14)	0.2985838444333054
  (0, 15)	-0.4222672119413615
  (0, 16)	1.2124077473000368
  (0, 17)	0.3251306212448608
  (0, 18)	0.30859731657625683
  (0, 19)	-0.7350191573217426
  (0, 20)	0.7917824386038563
  (0, 18440)	1.0
  (0, 70585)	1.0
  (0, 70587)	1.0
  (0, 70590)	1.0
  :	:
  (70563, 15)	0.6242732817928245
  (70563, 16)	-0.8675663804268393
  (70563, 17)	0.7592574419558187
  (70563, 18)	-0.2896771489649326
  (70563, 19)	1.3154865583765594
  (70563, 20)